# Generación y subida de datos IoT a S3

Este notebook genera datos fake de sensores IoT y los sube a un bucket S3 privado.

**Requisitos:**
- Tener configurado `aws configure` o ejecutar en un entorno con credenciales AWS
- Tener creado el bucket S3

## 1. Configuración

In [1]:
# Instalar dependencias desde requirements.txt (ejecutar solo la primera vez)
!pip install -r ../requirements.txt

  Using cached streamlit-1.54.0-py3-none-any.whl.metadata (9.8 kB)
  Using cached altair-6.0.0-py3-none-any.whl.metadata (11 kB)
  Using cached blinker-1.9.0-py3-none-any.whl.metadata (1.6 kB)
  Using cached cachetools-6.2.6-py3-none-any.whl.metadata (5.6 kB)
  Using cached gitpython-3.1.46-py3-none-any.whl.metadata (13 kB)
  Using cached pydeck-0.9.1-py2.py3-none-any.whl.metadata (4.1 kB)
  Using cached tenacity-9.1.4-py3-none-any.whl.metadata (1.2 kB)
  Using cached toml-0.10.2-py2.py3-none-any.whl.metadata (7.1 kB)
  Using cached watchdog-6.0.0-py3-none-win_amd64.whl.metadata (44 kB)
  Using cached gitdb-4.0.12-py3-none-any.whl.metadata (1.2 kB)
  Using cached smmap-5.0.2-py3-none-any.whl.metadata (4.3 kB)
Using cached streamlit-1.54.0-py3-none-any.whl (9.1 MB)
Using cached altair-6.0.0-py3-none-any.whl (795 kB)
Using cached blinker-1.9.0-py3-none-any.whl (8.5 kB)
Using cached cachetools-6.2.6-py3-none-any.whl (11 kB)
Using cached gitpython-3.1.46-py3-none-any.whl (208 kB)
Using cac

In [2]:
import os
from dotenv import load_dotenv

# Cargar variables desde .env (en la raiz del proyecto)
load_dotenv("../.env")

# Configuracion S3
AWS_REGION = os.getenv("AWS_REGION", "us-east-1")
S3_BUCKET = os.getenv("S3_BUCKET", "")
S3_KEY = os.getenv("S3_KEY", "")

# Credenciales AWS
AWS_ACCESS_KEY_ID = os.getenv("AWS_ACCESS_KEY_ID", "")
AWS_SECRET_ACCESS_KEY = os.getenv("AWS_SECRET_ACCESS_KEY", "")
AWS_SESSION_TOKEN = os.getenv("AWS_SESSION_TOKEN", "")

# Parametros de generacion
NUM_SENSORES = 5
REGISTROS_POR_SENSOR = 20

print("Variables cargadas desde .env")

Variables cargadas desde .env


## 2. Importar librerías

In [3]:
import json
import random
from datetime import datetime, timedelta

import boto3

print("Librerias importadas correctamente")

Librerias importadas correctamente


## 3. Generar datos fake

In [4]:
# Ubicaciones de ejemplo
UBICACIONES = [
    {"ciudad": "Madrid", "lat": 40.4168, "lon": -3.7038},
    {"ciudad": "Barcelona", "lat": 41.3851, "lon": 2.1734},
    {"ciudad": "Valencia", "lat": 39.4699, "lon": -0.3763},
    {"ciudad": "Sevilla", "lat": 37.3891, "lon": -5.9845},
    {"ciudad": "Bilbao", "lat": 43.2630, "lon": -2.9350},
]

def generate_sensor_data():
    """Genera datos fake de sensores IoT."""
    data = []
    base_time = datetime.utcnow() - timedelta(hours=2)
    
    for i in range(NUM_SENSORES):
        sensor_id = f"sensor_{i+1:02d}"
        ubicacion = UBICACIONES[i % len(UBICACIONES)]
        
        for j in range(REGISTROS_POR_SENSOR):
            timestamp = base_time + timedelta(minutes=j * 5)
            
            # Valores aleatorios
            temperature = round(random.uniform(15.0, 45.0), 2)
            co2 = round(random.uniform(300, 1200), 2)
            
            # Estado basado en temperatura
            if temperature > 40:
                state = "FAIL"
            elif temperature > 30:
                state = "WARN"
            else:
                state = "OK"
            
            record = {
                "timestamp": timestamp.isoformat(),
                "sensor_id": sensor_id,
                "sensor_state": state,
                "temperature_c": temperature,
                "co2_ppm": co2,
                "lat": ubicacion["lat"],
                "lon": ubicacion["lon"],
            }
            data.append(record)
    
    return data

# Generar datos
sensor_data = generate_sensor_data()
print(f"Generados {len(sensor_data)} registros")
print(f"Sensores: {NUM_SENSORES}")
print(f"Registros por sensor: {REGISTROS_POR_SENSOR}")

Generados 100 registros
Sensores: 5
Registros por sensor: 20


C:\Users\juanb\AppData\Local\Temp\ipykernel_28356\2780360602.py:13: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  base_time = datetime.utcnow() - timedelta(hours=2)


## 4. Vista previa de los datos

In [5]:
# Mostrar primeros 3 registros
print("Primeros 3 registros:")
print(json.dumps(sensor_data[:3], indent=2))

Primeros 3 registros:
[
  {
    "timestamp": "2026-02-22T17:53:58.031218",
    "sensor_id": "sensor_01",
    "sensor_state": "FAIL",
    "temperature_c": 42.58,
    "co2_ppm": 331.19,
    "lat": 40.4168,
    "lon": -3.7038
  },
  {
    "timestamp": "2026-02-22T17:58:58.031218",
    "sensor_id": "sensor_01",
    "sensor_state": "FAIL",
    "temperature_c": 44.1,
    "co2_ppm": 381.94,
    "lat": 40.4168,
    "lon": -3.7038
  },
  {
    "timestamp": "2026-02-22T18:03:58.031218",
    "sensor_id": "sensor_01",
    "sensor_state": "OK",
    "temperature_c": 26.18,
    "co2_ppm": 664.18,
    "lat": 40.4168,
    "lon": -3.7038
  }
]


In [6]:
# Estadisticas
estados = {}
for r in sensor_data:
    estado = r["sensor_state"]
    estados[estado] = estados.get(estado, 0) + 1

print("Distribucion de estados:")
for estado, count in sorted(estados.items()):
    print(f"  {estado}: {count}")

Distribucion de estados:
  FAIL: 19
  OK: 52
  WARN: 29


## 5. Guardar localmente

In [7]:
# Guardar en archivo local
local_file = "sensores.json"
with open(local_file, "w", encoding="utf-8") as f:
    json.dump(sensor_data, f, indent=2, ensure_ascii=False)

print(f"Guardado en: {local_file}")

Guardado en: sensores.json


## 6. Subir a S3

In [8]:
# Verificar configuracion
print(f"Region: {AWS_REGION}")
print(f"Bucket: {S3_BUCKET}")
print(f"Key: {S3_KEY}")
print()

# Validar credenciales
if not AWS_ACCESS_KEY_ID or not AWS_SECRET_ACCESS_KEY:
    print("ERROR: Falta AWS_ACCESS_KEY_ID o AWS_SECRET_ACCESS_KEY")
    print("Obtenlas de: AWS Academy -> AWS Details -> Show")
else:
    print("Credenciales configuradas OK")

Region: us-east-1
Bucket: juancho-sensores
Key: data/sensores/iabd11_sensores.json

Credenciales configuradas OK


In [9]:
# Subir a S3 (usando credenciales configuradas)
s3 = boto3.client(
    "s3",
    region_name=AWS_REGION,
    aws_access_key_id=AWS_ACCESS_KEY_ID,
    aws_secret_access_key=AWS_SECRET_ACCESS_KEY,
    aws_session_token=AWS_SESSION_TOKEN if AWS_SESSION_TOKEN else None
)

try:
    s3.upload_file(local_file, S3_BUCKET, S3_KEY)
    print(f"Subido correctamente a: s3://{S3_BUCKET}/{S3_KEY}")
except Exception as e:
    print(f"Error al subir: {e}")

Subido correctamente a: s3://juancho-sensores/data/sensores/iabd11_sensores.json


## 7. Verificar subida

In [10]:
# Listar objetos en el bucket (usa el cliente s3 ya configurado)
try:
    response = s3.list_objects_v2(Bucket=S3_BUCKET, Prefix="data/sensores/")
    print("Objetos en data/sensores/:")
    for obj in response.get("Contents", []):
        print(f"  - {obj['Key']} ({obj['Size']} bytes)")
except Exception as e:
    print(f"Error al listar: {e}")

Objetos en data/sensores/:
  - data/sensores/iabd11_sensores.json (21148 bytes)
